In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from pathlib import Path


In [22]:
import re
import json
from pathlib import Path
import numpy as np
from PIL import Image



def reconstruct_image_and_label_from_vnnlib(
    vnnlib_path: str,
    out_dir: str,
    *,
    # If your vnnlib inputs are normalized (common), pass mean/std to denormalize before saving PNG.
    # For CIFAR-100 typical mean/std (sometimes used): mean=(0.5071,0.4867,0.4408), std=(0.2675,0.2565,0.2761)
    # BUT benchmarks vary, so only set if you're sure.
    mean=None,   # e.g. (0.5071, 0.4867, 0.4408)
    std=None,    # e.g. (0.2675, 0.2565, 0.2761)
    # How to reshape if N=3072:
    cifar_layout="CHW",  # "CHW" or "HWC"
    save_float_npy=True,
):
    """
    Parse a .vnnlib file, reconstruct a nominal input image (midpoint of bounds),
    infer label from output constraints, and save (png + npy + meta.json).

    Returns: (image_array_float, label_or_none, paths_dict)
    """
    vnnlib_path = Path(vnnlib_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    text = vnnlib_path.read_text(errors="ignore")

    # -----------------------------
    # 1) Parse X bounds
    # -----------------------------
    # Matches: (assert (<= X_123 0.5)) and (assert (>= X_123 -0.5))
    re_le = re.compile(r"\(assert\s*\(\s*<=\s*X_(\d+)\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\s*\)\s*\)")
    re_ge = re.compile(r"\(assert\s*\(\s*>=\s*X_(\d+)\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\s*\)\s*\)")

    lb = {}  # i -> value
    ub = {}  # i -> value

    for m in re_ge.finditer(text):
        i = int(m.group(1))
        v = float(m.group(2))
        # keep the tightest (largest) lower bound if repeated
        lb[i] = max(lb.get(i, -np.inf), v)

    for m in re_le.finditer(text):
        i = int(m.group(1))
        v = float(m.group(2))
        # keep the tightest (smallest) upper bound if repeated
        ub[i] = min(ub.get(i, np.inf), v)

    if not lb and not ub:
        raise ValueError("No X_i bounds found in this vnnlib.")

    all_idx = sorted(set(lb.keys()) | set(ub.keys()))
    n = all_idx[-1] + 1  # assumes indices start at 0 and are contiguous-ish

    lb_arr = np.full((n,), -np.inf, dtype=np.float32)
    ub_arr = np.full((n,),  np.inf, dtype=np.float32)

    for i, v in lb.items():
        if i < n: lb_arr[i] = v
    for i, v in ub.items():
        if i < n: ub_arr[i] = v

    # sanity: if some indices are missing both bounds, you’ll get infs
    missing = np.isinf(lb_arr) | np.isinf(ub_arr)
    if missing.any():
        # try to handle one-sided bounds: if only lb or only ub exists, midpoint is not defined.
        # We'll fill missing side with the existing side (so midpoint = that value) and report it.
        only_lb = (~np.isinf(lb_arr)) & (np.isinf(ub_arr))
        only_ub = (np.isinf(lb_arr)) & (~np.isinf(ub_arr))
        both_missing = (np.isinf(lb_arr)) & (np.isinf(ub_arr))

        ub_arr[only_lb] = lb_arr[only_lb]
        lb_arr[only_ub] = ub_arr[only_ub]

        if both_missing.any():
            # Can't reconstruct these; set to 0 and report.
            lb_arr[both_missing] = 0.0
            ub_arr[both_missing] = 0.0

    x_mid = (lb_arr + ub_arr) / 2.0  # reconstructed nominal input (likely normalized)

    # -----------------------------
    # 2) Infer label from output constraints
    # -----------------------------
    # Your format: (and (>= Y_k Y_31)) repeated for many k -> reference label is 31
    # We'll detect the most common RHS in patterns (>= Y_a Y_b)
    re_ycomp = re.compile(r"\(>=\s*Y_(\d+)\s+Y_(\d+)\)")
    rhs_counts = {}
    for m in re_ycomp.finditer(text):
        rhs = int(m.group(2))
        rhs_counts[rhs] = rhs_counts.get(rhs, 0) + 1

    label = None
    if rhs_counts:
        # Heuristic: label is the RHS that appears most
        label = max(rhs_counts.items(), key=lambda kv: kv[1])[0]

    # -----------------------------
    # 3) Reshape into image
    # -----------------------------
    img = x_mid.copy()

    # CIFAR case: 3*32*32 = 3072
    if n == 3072:
        if cifar_layout.upper() == "CHW":
            img = img.reshape(3, 32, 32)         # (C,H,W)
            img_for_png = np.transpose(img, (1, 2, 0))  # -> (H,W,C)
        else:
            img = img.reshape(32, 32, 3)         # (H,W,C)
            img_for_png = img
    else:
        # generic: try square grayscale
        side = int(round(np.sqrt(n)))
        if side * side == n:
            img = img.reshape(side, side)
            img_for_png = img
        else:
            # fallback: store as flat
            img_for_png = img

    # -----------------------------
    # 4) Denormalize (optional) + save PNG
    # -----------------------------
    img_png = img_for_png.astype(np.float32)

    denorm_used = False
    if mean is not None and std is not None:
        mean = np.array(mean, dtype=np.float32)
        std = np.array(std, dtype=np.float32)

        # apply only if it looks like an image with channels
        if img_png.ndim == 3 and img_png.shape[-1] in (1, 3):
            img_png = (img_png * std) + mean
            denorm_used = True

    # Convert to uint8 safely for PNG:
    # If values are already [0,1] -> multiply by 255.
    # If values look like [-something, +something], we clip to [0,1] after optional denorm.
    if img_png.ndim in (2, 3):
        img_u8 = np.clip(img_png, 0.0, 1.0)
        img_u8 = (img_u8 * 255.0 + 0.5).astype(np.uint8)
        if img_u8.ndim == 2:
            pil = Image.fromarray(img_u8, mode="L")
        else:
            pil = Image.fromarray(img_u8, mode="RGB")
        png_path = out_dir / f"{label}.png"
        pil.save(png_path)
    else:
        png_path = None

    # -----------------------------
    # 5) Save NPY + metadata
    # -----------------------------
    npy_path = out_dir / f"{label}.npy"
    if save_float_npy:
        np.save(npy_path, img.astype(np.float32))
    else:
        np.save(npy_path, img_png.astype(np.float32))

    meta = {
        "vnnlib": str(vnnlib_path),
        "n_inputs": int(n),
        "inferred_label": label,
        "rhs_label_vote_counts": rhs_counts,
        "cifar_layout": cifar_layout,
        "denormalize_mean": mean if isinstance(mean, (list, tuple)) else (mean.tolist() if mean is not None else None),
        "denormalize_std": std if isinstance(std, (list, tuple)) else (std.tolist() if std is not None else None),
        "denorm_used": denorm_used,
        "png_saved": str(png_path) if png_path else None,
        "npy_saved": str(npy_path),
        "notes": (
            "Image is reconstructed as midpoint of bounds. "
            "PNG is clipped to [0,1] (after optional denorm)."
        ),
    }
    meta_path = out_dir / f"{label}.json"
    meta_path.write_text(json.dumps(meta, indent=2))

    paths = {"png": str(png_path) if png_path else None, "npy": str(npy_path), "meta": str(meta_path)}
    return img.astype(np.float32), label, paths

In [23]:
VNNLIB_PATH = "vnnlib/CIFAR100_resnet_large_prop_idx_383_sidx_3388_eps_0.0039.vnnlib"  # <-- change this
OUT_DIR = "decoded_vnnlib_output"  # folder to save recon.png, recon.npy, meta.json

img, label, paths = reconstruct_image_and_label_from_vnnlib(
    VNNLIB_PATH,
    out_dir=OUT_DIR,
    mean=(0.5071, 0.4867, 0.4408),
    std=(0.2675, 0.2565, 0.2761),
    cifar_layout="CHW",
)

print("Inferred label:", label)
print("Saved files:", paths)

Inferred label: 81
Saved files: {'png': 'decoded_vnnlib_output/81.png', 'npy': 'decoded_vnnlib_output/81.npy', 'meta': 'decoded_vnnlib_output/81.json'}
